In [1]:
# To be able to make edits to repo without having to restart notebook
%load_ext autoreload
%autoreload 2

In [2]:
import sys, os
sys.path.append(os.path.dirname(os.getcwd()))

import json
import pathlib
import pandas as pd
import mongomock
import xarray as xr
import cv2
from scipy import signal
import numpy as np
from fsspec.implementations.local import LocalFileSystem
from pathlib import Path
import matplotlib.pyplot as plt
from signalstore import UnitOfWorkProvider
from pymongo import MongoClient
import shutil


In [3]:
# Functions

def deserialize_dataarray(data_object):
        """Deserializes a data object.
        Arguments:
            data_object {dict} -- The data object to deserialize.
        Returns:
            dict -- The deserialized data object.
        """
        attrs = data_object.attrs.copy()
        for key, value in attrs.items():
            if isinstance(value, str):
                value = value.replace("'", '"')
                if value.lower() == 'true':
                    attrs[key] = True
                elif value.lower() == 'false':
                    attrs[key] = False
                elif value.lower() == 'none':
                    attrs[key] = None
                elif value.startswith('{'):
                    attrs[key] = json.loads(value)
            if isinstance(value, np.ndarray):
                attrs[key] = value.tolist()
        data_object.attrs = attrs
        return data_object


In [12]:
# Mock DB client
mongo_uri = os.environ.get("MONGO_URI", "mongodb://localhost:27017")
mongo_client = MongoClient(mongo_uri)
# -------------------------------------------------------------------

# 1) Define your container‐side data root (where "input" and "internal" are siblings)
# DATA_ROOT     = Path.cwd().parent / "data"
# INPUT_DIR     = DATA_ROOT / "input"
# INTERNAL_DIR  = DATA_ROOT / "internal"
# INTERNAL_DIR.mkdir(parents=True, exist_ok=True)


#filesystem: clear the internal data directory
tmpdir = Path.cwd().parent / "data" / "internal"
if not tmpdir.exists():
    tmpdir.mkdir(parents=True, exist_ok=True)
# for file_path in tmpdir.glob("*"):
#     if file_path.is_file():
#         file_path.unlink()
#         print(f"Deleted: {file_path}")


tmpdir = pathlib.Path(tmpdir)
filesystem = LocalFileSystem(root=str(tmpdir))
# filesystem = LocalFileSystem(root=str(DATA_ROOT))

# clear tmpdir
# for file in filesystem.ls(tmpdir):
#     print(file)
    # filesystem.rm(file)

# Empty memory store for demo
memory_store = dict()

# Get uow to act on database
uow_provider = UnitOfWorkProvider(mongo_client, filesystem, memory_store)
unit_of_work = uow_provider(str(tmpdir.name))
shutil.rmtree("./internal", ignore_errors=True)
with unit_of_work as uow:
    uow.data._data._directory = str(tmpdir)


# Get raw_property_models
property_models_path = Path.cwd().parent / r"tests" / r"data" / r"valid_data" / r"models" / r"property_models.json"
with open(property_models_path, 'r') as file:
    raw_property_models = json.load(file)

# Get raw_metamodels
metamodels_dir = Path.cwd().parent / r"tests" / r"data" / r"valid_data" / r"models" / r"metamodels"
metamodel_filepaths = list(metamodels_dir.glob("*.json"))
raw_metamodels = []
for filepath in metamodel_filepaths:
    with open(filepath, 'r') as file:
        raw_metamodels.append(json.load(file))

# Get raw_data_models
data_models_dir = Path.cwd().parent / r"tests" / r"data" / r"valid_data" / r"models" / r"data_models"
data_model_filepaths = list(data_models_dir.glob("*.json"))
raw_data_models = []
for filepath in data_model_filepaths:
    with open(filepath, 'r') as file:
        raw_data_models.append(json.load(file))

# Get raw_records
netcdf_dir = Path.cwd().parent / r"data" / r"input"
records_files = list(netcdf_dir.glob("*.xlsx"))
raw_records = []
for filepath in records_files:
        file = pd.read_excel(filepath, engine='openpyxl', dtype=str)
        file_json = file.to_json(orient="records")
        records = json.loads(file_json)
        raw_records.extend(records)

# Get dataarrays

netcdf_files = list(netcdf_dir.glob("*.nc"))
dataarrays = []
for filepath in netcdf_files:
    # print(filepath)
    dataarray = xr.open_dataarray(filepath)
    dataarray = deserialize_dataarray(dataarray)
    dataarrays.append(dataarray)

# Add property models, metamodels, data models, and records
with unit_of_work as uow:
    for property_model in raw_property_models:
        if not uow.domain_models.exists(property_model['schema_name']):
            print(f"Adding {property_model['schema_name']} to Mongo.")
            uow.domain_models.add(property_model)
        else:
            print(f"Model {property_model['schema_name']} already exists; skipping.")
    for metamodel in raw_metamodels:
        if not uow.domain_models.exists(metamodel['schema_name']):
            print(f"Adding {metamodel['schema_name']} to Mongo.")
            uow.domain_models.add(metamodel)
        else:
            print(f"Model {metamodel['schema_name']} already exists; skipping.")
    for data_model in raw_data_models:
        if not uow.domain_models.exists(data_model['schema_name']):
            print(f"Adding {data_model['schema_name']} to Mongo.")
            uow.domain_models.add(data_model)
        else:
            print(f"Model {data_model['schema_name']} already exists; skipping.")
    for record in raw_records:
        if not record.get("has_file"):
            schema_ref = record.get("schema_ref")
            data_name = record.get("data_name")
            version_timestamp = record.get("version_timestamp")  # Could be None if not defined
    
            if not uow.data.exists(schema_ref, data_name, version_timestamp):
                uow.data.add(record)
                print(f"Adding {schema_ref}/{data_name} to Mongo.")
            else:
                # You might update or log that the record already exists
                print(f"Record with schema_ref {schema_ref} and data_name {data_name} already exists.")
    for dataarray in dataarrays:
        # Skip dataarrays with schema_ref of "test"
        if dataarray.attrs.get("schema_ref") == "test":
            continue    
        # Extract the required identifying attributes from the dataarray's attrs
        schema_ref = dataarray.attrs.get("schema_ref")
        data_name = dataarray.attrs.get("data_name")
    
        # Check whether the data array already exists in the repository.
        # Note: if you're using a specific data adapter, pass it in; otherwise, it will use the default.
        if not uow.data.has_file(schema_ref, data_name):
            print(f"Adding {schema_ref}/{data_name} to Mongo.")
            uow.data.add(dataarray)
        # if not uow.data.exists(schema_ref, data_name, version_timestamp) \
        # or not uow.data.has_file(schema_ref, data_name, version_timestamp):
        #     print(f"Adding {schema_ref}/{data_name} to Mongo and file system.")
        #     uow.data._data.add(dataarray)
        else:
            print(f"{schema_ref}/{data_name} already in Mongo _and_ file exists; skipping.")

    uow.commit()

Adding version_timestamp to Mongo.
Adding schema_ref to Mongo.
Adding schema_type to Mongo.
Adding schema_name to Mongo.
Adding schema_title to Mongo.
Adding schema_description to Mongo.
Adding data_name to Mongo.
Adding time_of_save to Mongo.
Adding time_of_removal to Mongo.
Adding record_type to Mongo.
Adding json_schema to Mongo.
Adding has_file to Mongo.
Adding unit_of_measure to Mongo.
Adding dimension_of_measure to Mongo.
Adding acquisition to Mongo.
Adding acquisition_date to Mongo.
Adding import_date to Mongo.
Adding acquisition_notes to Mongo.
Adding data_dimensions to Mongo.
Adding shape to Mongo.
Adding dtype to Mongo.
Adding session_description to Mongo.
Adding session_date to Mongo.
Adding session_time to Mongo.
Adding session_duration to Mongo.
Adding session_notes to Mongo.
Adding data_ref to Mongo.
Adding start_time to Mongo.
Adding duration to Mongo.
Adding duration_unit to Mongo.
Adding animal_species to Mongo.
Adding age to Mongo.
Adding age_unit to Mongo.
Adding age

DataRepositoryValidationError: 

DataRepositoryValidationError
-------------------

Property Name: sampling_rate

Message: 48000 is not of type 'string'

Instance: 48000

Path: deque([])

Relative Path: deque([])

Absolute Path: deque([])

Schema Path: deque(['type'])

Local Schema: {'type': 'string'}

Args: ("48000 is not of type 'string'", <unset>, (), None, (), <unset>, <unset>, <unset>, (), None)

Cause: None

Context: []

Validator: type

Validator Value: string

Record: {'session_data_ref': {'schema_ref': 'session', 'data_name': '1-13_20210510_110723'}, 'animal_data_ref': {'schema_ref': 'animal', 'data_name': '1-13'}, 'probe_data_ref': {'schema_ref': 'probe', 'data_name': '1'}, 'has_file': True, 'schema_ref': 'tet_waveforms', 'data_name': '1-13_1_20210510_110723', 'data_dimensions': ['spike_idx', 'channel', 'sample'], 'dimension_of_measure': '[charge]', 'sampling_rate': 48000, 'duration': 1200}

Full Schema: {'type': 'string'}



In [ ]:
def view_query_factory(uow, query, cell_id=None, exclude=None):
    """ 
    Example custom use case by user to get a dictionary of xarray datasets 

    Arguments:

    uow {UnitOfWork} -- Unit of work instance
    query {dict} -- Query to filter data with schema_ref and data_name as keys
    cell_id {numeric} -- Optional cell id to filter data for a single cell
    exclude {list} -- Optional list of schema_refs to exclude from query

    Outputs:

    query_dataset {dict} -- Dictionary of xarray datasets

    Example use case:

    To query across multiple sets of data and organize neural data while excluding behavioral data

    E.g. can get dictionary of multiple session with spike times/labels/waveforms organized. Can filter by session, tetrode, animal and other relevant schema. 
    Can also restrict to one cell over time and exclude schema like position which are behavioral recordings.

    """
    # first get query data
    query_data = uow.data.find(query)

    # empty output dict
    session_data = {}

    # go through each schema obtained by the query
    for _, d in enumerate(query_data):

        schema_ref = d['schema_ref'] # e.g. session, animal_position, spike_times, spike_labels
        data_name = d['data_name'] # data_name is id that references schema, e.g. 'ANT-133a-4_20180517_133858' for session or '3' for probe id
        # data_names are custom ids with components determined by user to ensure unique data_names 

        # custom addition to exclude schema_ref = animak_position
        # animal_position is consistent across tetrodes in a recording session and should be excluded
        # when querying neural data 
        if exclude is not None and schema_ref in exclude:
                continue

        # retrieve data and add to dictionary ensuring no duplicates 
        # also organizes different schema_ref with same data_name (e.g. spike times and labels for same session)
        if data_name not in session_data:
            session_data[data_name] = {}
        retrieved_data = uow.data.get(schema_ref, data_name)
        assert schema_ref not in session_data[data_name], f"Data already exists for {schema_ref} in {data_name}"
        session_data[data_name][schema_ref] = retrieved_data

    # organize sets of schema_refs into datasets for hte unique sessions collected
    # index is data_name of session
    query_dataset = {name: xr.Dataset(data) for name, data in session_data.items()}
    # query_dataset = [xr.Dataset(data) for name, data in session_data.items()]
    
    # optionally filter each dataset by a cell_id 
    # only valid if cell_ids are matched across sessions (true in our data)
    if cell_id is not None:
        query_dataset = {name: _filter_cell_data(data, cell_id) for name, data in query_dataset.items()}
        
    return query_dataset

# helper fxn to only select single cell label
def _filter_cell_data(data, cell_id):
    """Get data for a single cell"""
    filtered_data = data.where(data.spike_labels == int(cell_id), drop=True)
    return filtered_data



In [ ]:
import numpy as np

def rgb2gray(rgb):
    r, g, b = rgb[:, :, 0], rgb[:, :, 1], rgb[:, :, 2]
    gray = 0.2989 * r + 0.5870 * g + 0.1140 * b
    return gray
def speed2D(x, y, t):
    """calculates an averaged/smoothed speed"""

    N = len(x)
    v = np.zeros((N, 1))

    for index in range(1, N-1):
        v[index] = np.sqrt((x[index + 1] - x[index - 1]) ** 2 + (y[index + 1] - y[index - 1]) ** 2) / (
        t[index + 1] - t[index - 1])

    v[0] = v[1]
    v[-1] = v[-2]

    return v


def centreBox(posx, posy):
    """
    Computes the centre of the box defined by posx and posy.

    Parameters:
      posx: array-like, x-coordinates of the points.
      posy: array-like, y-coordinates of the points.

    Returns:
      list: [x, y] coordinates of the centre.
    """
    maxX = np.max(posx)
    minX = np.min(posx)
    maxY = np.max(posy)
    minY = np.min(posy)
    
    # Define the corners of the box
    NE = [maxX, maxY]
    NW = [minX, maxY]
    SW = [minX, minY]
    SE = [maxX, minY]
    
    centre = findCentre(NE, NW, SW, SE)
    return centre

def findCentre(NE, NW, SW, SE):
    """
    Calculates the centre of the box from the corner coordinates.
    The centre is the intersection of the diagonals.

    Parameters:
      NE, NW, SW, SE: list or array containing the [x, y] coordinates of the corners.
      
    Returns:
      list: [x, y] coordinates of the centre.
    """
    # Compute slopes for the diagonals
    a = (NE[1] - SW[1]) / (NE[0] - SW[0])  # slope for NE-SW diagonal
    b = (SE[1] - NW[1]) / (SE[0] - NW[0])  # slope for SE-NW diagonal
    c = SW[1]
    d = NW[1]
    x = (d - c + a * SW[0] - b * NW[0]) / (a - b)
    y = a * (x - SW[0]) + c
    return [x, y]

import numpy as np
from skimage.measure import label, regionprops

def bwarea_old(binary_image: np.ndarray) -> float:
    """
    Estimate the area of a binary object similar to MATLAB's bwarea.

    MATLAB's bwarea computes the area as:
        area = N + 0.5*P + 0.25*V
    where:
      - N is the number of object pixels,
      - P is the perimeter (here computed using regionprops),
      - V is the number of vertices (assumed 4 for a simple closed shape).
      
    For example, for a 40x40 filled square:
        N = 1600, P = 160, and V = 4,
        so area = 1600 + 0.5*160 + 0.25*4 = 1681.
    
    Parameters:
      binary_image: np.ndarray
          Input binary image (0/1 or bool).
    
    Returns:
      float: The estimated area.
    """
    # Ensure the image is boolean.
    bw = binary_image.astype(bool)
    
    # N: number of object pixels.
    N = np.sum(bw)
    
    # Compute perimeter using regionprops.
    # Note: regionprops uses a Crofton perimeter estimator.
    labeled = label(bw)
    props = regionprops(labeled)
    if len(props) > 0:
        P = props[0].perimeter
    else:
        P = 0
    
    # Assume V = 4 for a simple shape (e.g. a square).
    V = 4 if N > 0 else 0
    
    area = N + 0.5 * P + 0.25 * V
    return area


from skimage.measure import find_contours

def bwarea(binary_image: np.ndarray) -> float:
    """
    MATLAB's bwarea computes the area as:
        area = N + 0.5*P + 0.25*V
    where:
      - N is the number of object pixels,
      - P is the perimeter (computed along the actual object boundary),
      - V is the number of vertices (assumed 4 for a simple closed shape).
    
    Parameters:
      binary_image: np.ndarray
          Input binary image (0/1 or bool).
    
    Returns:
      float: The estimated area.
    """
    # Ensure the image is boolean.
    bw = binary_image.astype(bool)
    
    # N: number of object pixels.
    N = np.sum(bw)
    
    # Compute perimeter by extracting the contour at level 0.5,
    # which gives the actual continuous boundary length.
    contours = find_contours(bw.astype(float), level=0.5)
    if contours:
        # Use the longest contour if there are several.
        longest = max(contours, key=lambda c: c.shape[0])
        # Compute the Euclidean length along the contour.
        P = np.sum(np.sqrt(np.sum(np.diff(longest, axis=0)**2, axis=1)))
    else:
        P = 0
    
    # Assume V = 4 for a simple shape (e.g. a square).
    V = 4 if N > 0 else 0
    
    area = N + 0.5 * P + 0.25 * V
    return area


In [ ]:
from skimage.color import rgb2gray
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable


def plot_bin_metrics_heatmaps(bin_coverage_png, bin_distance_png, n_bins, bin_coverage, bin_distances):
    """    
    Parameters:
      dimensions: array-like, [xmin, xmax, ymin, ymax] of the arena.
      n_bins: int, number of bins per dimension (e.g. 4 for a 4x4 grid).
      bin_coverage: (n_bins x n_bins) array of percentage coverage values.
      bin_distances: (n_bins x n_bins) array of total distance values.
      output_path_prefix: str, prefix for the output file names.
    
    The function saves two PNG files:
      - {output_path_prefix}_coverage_heatmap.png
      - {output_path_prefix}_distance_heatmap.png
    """

    # --- Coverage Heatmap ---
    fig_cov, ax_cov = plt.subplots(figsize=(n_bins*2, n_bins*2))  # Adjust size based on number of bins
    # origin='lower' to have bin[0,0] at the bottom-left corner
    im_cov = ax_cov.imshow(bin_coverage, cmap='Greys', origin='lower')
    ax_cov.set_title("Bin Coverage Heatmap (%)")
    # Set tick labels to indicate bin positions (optional)
    ax_cov.set_xticks(np.arange(n_bins))
    ax_cov.set_yticks(np.arange(n_bins))
    ax_cov.set_xticklabels([f"{x+1:.1f}" for x in range(n_bins)])
    ax_cov.set_yticklabels([f"{y+1:.1f}" for y in range(n_bins)])
    divider = make_axes_locatable(ax_cov)
    cax = divider.append_axes("right", size="5%", pad=0.05)
    fig_cov.colorbar(im_cov, cax=cax, label="Coverage")
    plt.tight_layout()
    fig_cov.savefig(bin_coverage_png, bbox_inches='tight', pad_inches=0)
    plt.close(fig_cov)

    # --- Distance Heatmap ---
    fig_dist, ax_dist = plt.subplots(figsize=(n_bins*2, n_bins*2))  # Adjust size based on number of bins
    im_dist = ax_dist.imshow(bin_distances, cmap='Greys', origin='lower')
    ax_dist.set_title("Bin Distance Heatmap (meters)")
    ax_dist.set_xticks(np.arange(n_bins))
    ax_dist.set_yticks(np.arange(n_bins))
    ax_dist.set_xticklabels([f"{x+1:.1f}" for x in range(n_bins)])
    ax_dist.set_yticklabels([f"{y+1:.1f}" for y in range(n_bins)])
    divider = make_axes_locatable(ax_dist)
    cax = divider.append_axes("right", size="5%", pad=0.05)
    fig_dist.colorbar(im_dist, cax=cax, label="Distance")
    plt.tight_layout()
    fig_dist.savefig(bin_distance_png, bbox_inches='tight', pad_inches=0)
    plt.close(fig_dist)

    print(f"Heatmap images saved as:\n  Coverage: {bin_coverage_png}\n  Distance: {bin_distance_png}")



def plot_bin_metrics(dimensions, n_bins, bin_coverage, bin_distances, bin_metrics_png):
    """
    Create and save a plot of an arena divided into bins (grid) with text annotations
    in each bin showing the bin's coverage and distance metrics.
    
    Parameters:
      dimensions: array-like, [xmin, xmax, ymin, ymax] defining the arena.
      n_bins: int, number of bins along each axis (e.g., 4 for a 4x4 grid).
      bin_coverage: numpy array of shape (n_bins, n_bins) with percentage coverage per bin.
      bin_distances: numpy array of shape (n_bins, n_bins) with total distance per bin.
      output_path: str, the file path to save the plot.
    """
    xmin, xmax, ymin, ymax = dimensions
    # Define bin boundaries in coordinate space.
    x_bins = np.linspace(xmin, xmax, n_bins + 1)
    y_bins = np.linspace(ymin, ymax, n_bins + 1)
    
    # Create a figure and axes.
    fig, ax = plt.subplots(figsize=(20, 10.6066667), dpi=150)
    
    # Draw vertical grid lines.
    for x in x_bins:
        ax.axvline(x=x, color='k', lw=1)
    # Draw horizontal grid lines.
    for y in y_bins:
        ax.axhline(y=y, color='k', lw=1)
    
    # Annotate each bin with its coverage and distance.
    for i in range(n_bins):
        for j in range(n_bins):
            # Compute the center of the bin.
            cx = (x_bins[i] + x_bins[i+1]) / 2
            cy = (y_bins[j] + y_bins[j+1]) / 2
            # Format the text: coverage (in %) and distance.
            text = f"Coverage: {bin_coverage[j, i]:.1f}%\nDistance: {bin_distances[j, i]:.2f}"
            ax.text(cx, cy, text, ha='center', va='center', fontsize=12, color='black')
    
    # Set the axis limits to the arena dimensions.
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_aspect('equal', adjustable='box')
    
    # Optionally hide the axis.
    ax.axis('off')
    
    # Adjust layout and save the figure.
    plt.tight_layout()
    plt.savefig(bin_metrics_png, bbox_inches='tight', pad_inches=0)
    plt.close(fig)
    print(f"Plot saved as {bin_metrics_png}")


def attribute_segment_to_bins(x1, y1, x2, y2, x_bins, y_bins):
    """
    Given a segment from (x1, y1) to (x2, y2) and grid boundaries (x_bins, y_bins),
    return a dictionary where keys are (bin_i, bin_j) and values are the length of
    the segment attributed to that bin.
    
    Parameters:
      x1, y1, x2, y2: float
          Endpoints of the segment.
      x_bins: 1D array of bin boundaries in x.
      y_bins: 1D array of bin boundaries in y.
    
    Returns:
      contributions: dict with keys (i,j) and values = subsegment length.
    """
    dx = x2 - x1
    dy = y2 - y1
    seg_length = np.sqrt(dx**2 + dy**2)
    if seg_length == 0:
        return {}
    
    # Start with t=0 and t=1
    t_vals = [0.0, 1.0]
    
    # Find t where the line crosses vertical boundaries (ignore the extreme boundaries)
    for X in x_bins[1:-1]:
        if dx != 0:
            t = (X - x1) / dx
            if 0 < t < 1:
                t_vals.append(t)
    
    # Find t where the line crosses horizontal boundaries
    for Y in y_bins[1:-1]:
        if dy != 0:
            t = (Y - y1) / dy
            if 0 < t < 1:
                t_vals.append(t)
    
    # Remove duplicates and sort
    t_vals = np.unique(t_vals)
    t_vals.sort()
    
    contributions = {}
    # For each subsegment, compute its midpoint and attribute its length to the appropriate bin.
    for i in range(len(t_vals) - 1):
        t_start = t_vals[i]
        t_end = t_vals[i+1]
        t_mid = (t_start + t_end) / 2.0
        xm = x1 + dx * t_mid
        ym = y1 + dy * t_mid
        # Determine which bin this midpoint falls into.
        # np.digitize returns indices starting at 1, so subtract 1.
        bin_i = np.digitize(xm, x_bins) - 1
        bin_j = np.digitize(ym, y_bins) - 1
        subseg_length = seg_length * (t_end - t_start)
        contributions[(bin_i, bin_j)] = contributions.get((bin_i, bin_j), 0) + subseg_length
    return contributions

def compute_bin_metrics(posx, posy, BWdfill_total, BWdfill_filled, dimensions, image_resolution, n_bins=4):
    """
    Divide the arena (square) into an n_bins x n_bins grid and compute for each bin:
      - percentage coverage: fraction (0-100%) of the bin area in the binary mask that is "filled"
      - total distance: sum of Euclidean distances for segments, proportionally attributed to bins if they cross boundaries.
    
    Parameters:
      posx, posy: 1D numpy arrays (in arena coordinate space) for animal positions.
      BWdfill_total: 2D binary numpy array representing total area (e.g., arena border mask).
      BWdfill_filled: 2D binary numpy array representing the filled area (e.g., coverage mask).
      dimensions: array-like, [xmin, xmax, ymin, ymax] of the arena.
      image_resolution: tuple (height, width) in pixels of the BWdfill image.
      n_bins: number of bins per dimension (default 4 for a 4x4 grid).
    
    Returns:
      coverage: a (n_bins x n_bins) numpy array with percentage coverage per bin.
      distance: a (n_bins x n_bins) numpy array with total distance traveled in each bin.
    """
    H, W = image_resolution
    xmin, xmax, ymin, ymax = dimensions
    dx = xmax - xmin
    dy = ymax - ymin
    
    # Create bin boundaries in coordinate space:
    x_bins = np.linspace(xmin, xmax, n_bins + 1)
    y_bins = np.linspace(ymin, ymax, n_bins + 1)
    
    # Initialize output arrays.
    coverage = np.zeros((n_bins, n_bins))
    distance = np.zeros((n_bins, n_bins))
    
    # --- Compute coverage per bin ---
    for i in range(n_bins):
        for j in range(n_bins):
            # Bin boundaries in coordinate space:
            x0, x1 = x_bins[i], x_bins[i+1]
            y0, y1 = y_bins[j], y_bins[j+1]
            
            # Map these boundaries to pixel coordinates:
            col0 = int(np.floor(((x0 - xmin) / dx) * (W - 1)))
            col1 = int(np.ceil(((x1 - xmin) / dx) * (W - 1)))
            row0 = int(np.floor(((y0 - ymin) / dy) * (H - 1)))
            row1 = int(np.ceil(((y1 - ymin) / dy) * (H - 1)))
            
            bin_total_area = bwarea(BWdfill_total[row0:row1, col0:col1])
            bin_total_filled = bwarea(BWdfill_filled[row0:row1, col0:col1])
            if bin_total_area > 0:
                bin_percent = (bin_total_filled / bin_total_area) * 100
            else:
                bin_percent = 0
            coverage[j, i] = bin_percent  # note: j for row, i for column

    # --- Compute total distance per bin with proportional attribution ---
    # Ensure posx, posy are 1D arrays.
    posx = posx.flatten()
    posy = posy.flatten()
    for k in range(len(posx) - 1):
        x1_pt, y1_pt = posx[k], posy[k]
        x2_pt, y2_pt = posx[k+1], posy[k+1]
        # Get contributions for the segment.
        contrib = attribute_segment_to_bins(x1_pt, y1_pt, x2_pt, y2_pt, x_bins, y_bins)
        for (i, j), d in contrib.items():
            # Check index bounds
            if 0 <= i < n_bins and 0 <= j < n_bins:
                # Convert to meters if positions are in centimeters (adjust divisor as needed)
                distance[j, i] += d / 100.0
    
    return coverage, distance


In [ ]:
## imports ##

import importlib.util
import sys
from scipy import ndimage
import numpy as np
import matplotlib.pyplot as plt
import imageio
from skimage import color
import skimage
import matplotlib
PROJECT_PATH = '/app/src/NSK-isolated-main'
sys.path.append(PROJECT_PATH)
print(PROJECT_PATH)
module_name = "animal_performance"


# Define file path
file_path = "/app/src/NSK-isolated-main/scripts/batch_map/animal_performance.py"

# Load module
spec = importlib.util.spec_from_file_location(module_name, file_path)
animal_performance = importlib.util.module_from_spec(spec)
sys.modules[module_name] = animal_performance
spec.loader.exec_module(animal_performance)

def is_circle(points, corners):
    """
    Determines if the arena is circular based on the points and corners.

    Parameters:
      points: np.ndarray
          Array of points (x, y) coordinates.
      corners: np.ndarray
          Array of corners (x, y) coordinates.

    Returns:
      bool: True if the arena is circular, False otherwise.
    """
    circle_bool = []
    for corner in range(4):
        if corner == 0:  # NE corner
            bool_val = (points[:, 0] >= 0) * (points[:, 1] >= 0)
        elif corner == 1:  # NW Corner
            bool_val = (points[:, 0] < 0) * (points[:, 1] >= 0)
        elif corner == 2:  # SW Corner
            bool_val = (points[:, 0] < 0) * (points[:, 1] < 0)
        else:  # SE corner
            bool_val = (points[:, 0] > 0) * (points[:, 1] < 0)
        
        current_points = points[bool_val, :]
        circle_bool.append(np.sum((current_points[:, 0] > corners[corner, 0]) * 
                                (current_points[:, 1] > corners[corner, 1])))

    return sum(circle_bool) < 1



## main function ##

def get_animal_performance(session_data, root_path, settings_dict):
    
    posx = session_data[:,0].to_numpy().squeeze()
    posy = session_data[:,1].to_numpy().squeeze()
    post = session_data[:,2].to_numpy().squeeze()
    print(posx.shape, posy.shape, post.shape)
    save_figures_directory = pathlib.Path.cwd().parent / r"data" / r"output"
    # plot_linewidth = 5.334
    plot_linewidth = 10.5
    # plot_linewidth = 10000 ## avg mice len is 2.25 according to the ppm calcluations
    if len(posx) == 0:
        print('There are no valid positions (all NaNs)')
    posx = posx.reshape(-1, 1)
    posy = posy.reshape(-1, 1)
    

    print(session_data)
    print('calculating the total distance for the .pos file: %s ' % session_data.attrs.get('session_data_ref')["data_name"])

    diffX = np.diff(posx, axis=0)
    diffY = np.diff(posy, axis=0)
    dist_sample = np.sqrt((diffX ** 2) + (diffY ** 2))
    total_distance = np.sum(dist_sample)
    total_distance = total_distance/100  # convert to meters
    print('Calculating coverage!')
    points = np.hstack((posx.reshape((len(posx), 1)), posy.reshape((len(posy), 1))))
    dimensions = np.array([np.amin(posx), np.amax(posx), np.amin(posy), np.amax(posy)])

    arena_fig = plt.figure(figsize=(20, 10.6066667), dpi=150)
    ax = arena_fig.add_subplot(111)
    ax.plot(posx, posy, 'r-', lw=plot_linewidth)
    ax.axis('off')
    

    bin_x = np.linspace(np.amin(posx), np.amax(posx), 11)
    bin_y = np.linspace(np.amin(posy), np.amax(posy), 11)
    corners = np.array([[bin_x[-2], bin_y[-2]], [bin_x[1], bin_y[-2]], 
                        [bin_x[1], bin_y[1]], [bin_x[-2], bin_y[1]]])
    
    plot_corners = False
    if plot_corners:
        # plots to see the test if it is a square or cirlce. I essentially
        # determine if there is data at the corners of the arena
        fig_corners = plt.figure()
        ax_corners = fig_corners.add_subplot(111)
        ax_corners.plot(posx, posy, 'r-')
        ax_corners.plot(corners[0:2, 0], corners[0:2, 1], 'g')
        ax_corners.plot(corners[1:3, 0], corners[1:3, 1], 'g')
        ax_corners.plot(corners[[0, 3], 0], corners[[0, 3], 1], 'g')
        ax_corners.plot(corners[[2, 3], 0], corners[[2, 3], 1], 'g')
        ax_corners.set_title('Corners of the bins')

    circle_bool = is_circle(points, corners)
    coverage_figure = plt.figure(figsize=(20, 10.6066667), dpi=150)
    ax_coverage = coverage_figure.add_subplot(111)
    if not circle_bool:
        print('Arena detected as being square')

        # Define bins for x and y
        bins = np.linspace(np.amin(posx), np.amax(posx), 20)
        bin_edges = np.hstack((bins[:-1].reshape((-1, 1)), bins[1:].reshape((-1, 1))))

        # Define rectangle for square arena
        rectangle_points = np.array([
            [np.amin(posx), np.amax(posy)],  # NW
            [np.amax(posx), np.amax(posy)],  # NE
            [np.amax(posx), np.amin(posy)],  # SE
            [np.amin(posx), np.amin(posy)],  # SW
            [np.amin(posx), np.amax(posy)]   # NW (closing the loop)
        ])

        border = ax_coverage.plot(rectangle_points[:, 0], rectangle_points[:, 1], 'b', lw=plot_linewidth / 10)
        # Plot arena boundaries
        ax_coverage.plot(rectangle_points[:, 0], rectangle_points[:, 1], 'b', lw=plot_linewidth / 10)
        ax_coverage.plot(posx, posy, 'r-', lw=plot_linewidth)

        # Set axis limits
        ax_coverage.set_xlim([min([dimensions[0], np.amin(rectangle_points[:, 0])]) - 0.5,
                                max([dimensions[1], np.amax(rectangle_points[:, 0])]) + 0.5])
        ax_coverage.set_ylim([min([dimensions[2], np.amin(rectangle_points[:, 1])]) - 0.5,
                                max([dimensions[3], np.amax(rectangle_points[:, 1])]) + 0.5])
        ax.set_xlim([min([dimensions[0], np.amin(rectangle_points[:, 0])]) - 0.5,
                        max([dimensions[1], np.amax(rectangle_points[:, 0])]) + 0.5])
        ax.set_ylim([min([dimensions[2], np.amin(rectangle_points[:, 1])]) - 0.5,
                                max([dimensions[3], np.amax(rectangle_points[:, 1])]) + 0.5])
                

    else:
        print('Arena detected as being circular')

        bins = np.linspace(np.amin(posx), np.amax(posx), 50)
        bin_edges = np.hstack((bins[:-1].reshape((-1, 1)), bins[1:].reshape((-1, 1))))

        # Compute radii
        radii = np.array([np.abs(np.amin(posx)), np.amax(posx), np.abs(np.amin(posy)), np.amax(posy)])
        for bin_value in range(len(bin_edges)):
            bin_bool = (posx >= bin_edges[bin_value, 0]) * (posx < bin_edges[bin_value, 1])
            if sum(bin_bool) == 0:
                continue

            posx_bin = posx[bin_bool]
            posy_bin = posy[bin_bool]
            max_val = np.amax(posy_bin)
            max_i = np.where(posy_bin == max_val)[0][0]
            min_val = np.amin(posy_bin)
            min_i = np.where(posy_bin == min_val)[0][0]
            append_radii = np.array([np.sqrt(max_val ** 2 + posx_bin[max_i] ** 2),
                                    np.sqrt(min_val ** 2 + posx_bin[min_i] ** 2)])
            radii = np.concatenate((radii, append_radii))

        # Generate circle for circular arena
        step = 0.001
        ang = np.arange(np.round((4 * np.pi + step) / step)) / (1 / step)
        xp, yp = animal_performance.circle_vals(0, 0, 2 * np.amax(radii), ang)
        border = ax_coverage.plot(xp, yp, 'b', lw=plot_linewidth / 10)
        ax_coverage.plot(xp, yp, 'b', lw=plot_linewidth / 10)
        ax_coverage.plot(posx, posy, 'r-', lw=plot_linewidth)
        

        ax_coverage.set_xlim([min([dimensions[0], np.amin(xp)]) - 0.5,
                                max([dimensions[1], np.amax(xp)]) + 0.5])
        ax_coverage.set_ylim([min([dimensions[2], np.amin(yp)]) - 0.5,
                                max([dimensions[3], np.amax(yp)]) + 0.5])
        ax.set_xlim([min([dimensions[0], np.amin(xp)]) - 0.5,
                                max([dimensions[1], np.amax(xp)]) + 0.5])
        ax.set_ylim([min([dimensions[2], np.amin(yp)]) - 0.5,
                                max([dimensions[3], np.amax(yp)]) + 0.5])
        

    # Save figure to file
    # save_path = "arena_plot.png"
    cover_png_total = os.path.join(save_figures_directory, '%s_total.png' % session_data.attrs.get('session_data_ref')["data_name"])
    ax_coverage.axis('off')
    # plt.subplots_adjust(left=0, right=1, top=1, bottom=0)  # Remove margins
    coverage_figure.patch.set_facecolor('white')
    # coverage_figure.savefig(cover_png_total, bbox_inches='tight', pad_inches=0, transparent=False)
    # coverage_figure.savefig(cover_png_total, dpi=150, pad_inches=0, transparent=False)
    coverage_figure.savefig(cover_png_total, dpi=150, pad_inches=0)

    # plt.savefig(save_path, bbox_inches='tight', dpi=300)
    print(f"Figure saved as {cover_png_total}")

    # Show the figure
    # plt.show()

    RGBA = imageio.imread(cover_png_total)
    RGB = color.rgba2rgb(RGBA)
    I = rgb2gray(RGB)
    I = np.round(I).astype('int32')

    BWs_x = ndimage.sobel(I, 0)  # horizontal derivative
    BWs_y = ndimage.sobel(I, 1)  # vertical derivative
    BWs = np.hypot(BWs_x, BWs_y)  # magnitude
    BWs *= 255.0 / np.amax(BWs) 
    BWsdil = ndimage.morphology.binary_dilation(BWs)
    BWdfill = ndimage.morphology.binary_fill_holes(BWsdil)
    total_area = bwarea(BWdfill)
    BWdfill_total = BWdfill.copy()
    border[0].remove()  # remove border 
    cover_png = os.path.join(save_figures_directory, '%s_coverage.png' % session_data.attrs.get('session_data_ref')["data_name"])
    # coverage_figure.savefig(cover_png, bbox_inches='tight')  # save figure without border
    ax.axis('off')
    arena_fig.savefig(cover_png, dpi=150, pad_inches=0, transparent=False)

    # reading in the positions without the arena trace
    RGBA = imageio.imread(cover_png)
    try:
        RGB = color.rgba2rgb(RGBA)
    except ValueError:
        RGB = RGBA
    I = rgb2gray(RGB)
    I = np.round(I).astype('int32')
    if np.amax(I) <= 1:
        # then the image was saved from numpy
        BWdfill = I < 1
    else:
        BWdfill = I < 255
    # finding the contours of the path so we can find the area

    coverage_fill_figure = plt.figure()
    ax_coverage_fill = coverage_fill_figure.add_subplot(111)
    # mng = plt.get_current_fig_manager()
    ax_coverage_fill.imshow(BWdfill, cmap=plt.cm.gray)
    plt.title("BWdfill2")
    filled_png = os.path.join(save_figures_directory, '%s_filled.png' % session_data.attrs.get('session_data_ref')["data_name"])
    coverage_fill_figure.savefig(filled_png, bbox_inches='tight')
    total_filled = bwarea(BWdfill)
    BWdfill_filled = BWdfill.copy()
    image_resolution = (1591, 3000)
    n_bins = 4
    bin_coverage, bin_distances = compute_bin_metrics(posx, posy, BWdfill_total,BWdfill_filled, dimensions, image_resolution, n_bins)
    bin_coverage_png = os.path.join(save_figures_directory, '%s_bin_coverage_heatmap.png' % session_data.attrs.get('session_data_ref')["data_name"])
    bin_distance_png = os.path.join(save_figures_directory, '%s_bin_distance_heatmap.png' % session_data.attrs.get('session_data_ref')["data_name"])
    bin_metrics_png = os.path.join(save_figures_directory, '%s_bin_metrics.png' % session_data.attrs.get('session_data_ref')["data_name"])
    plot_bin_metrics_heatmaps(bin_coverage_png, bin_distance_png, n_bins, bin_coverage, bin_distances)
    plot_bin_metrics(dimensions, n_bins, bin_coverage, bin_distances, bin_metrics_png)
    print('total area: %f' % total_area)
    print('total filled: %f' % total_filled)
    percent = (total_filled / total_area) * 100
    print('Coverage(Percent): %f' % (100*total_filled/total_area))

    speed = speed2D(posx, posy, post)

    print(f"total_area: {total_area}")
    print(f"total_filled: {total_filled}")
    print(f"percent: {percent}")
    print(f"total_distance: {total_distance}")
    print(f"speed: {speed}")

    return total_area, total_filled, percent, total_distance, speed, bin_distances, bin_coverage


In [ ]:
import os
import pandas as pd
import datetime
import tkinter as tk
from tkinter import filedialog
import time
from openpyxl.utils.cell import get_column_letter

# Example function signature for refactored code
def batch_map_via_queries(uow, session_queries, settings_dict, save_dir):


    csv_header = settings_dict['header']
    run_number = 1
    root_path = os.path.join(save_dir, 'Animal_Performance_Results')
    while os.path.isdir(root_path + str(run_number)):
        run_number += 1
    root_path += str(run_number)
    os.mkdir(root_path)

    # Initialize empty dataframe with columns
    headers = [k for k, v in csv_header.items() if v]
    headers_dict = dict()
    for i, header in enumerate(headers):
        headers_dict[header] = get_column_letter(i+1) ## Excel function to get let's say Column A, B, C etc.

    file_name = os.path.join(root_path, "ap_parameters.txt")
    with open(file_name, 'w') as f:
        for ky in settings_dict:
            f.write(str(ky) + ' is: ')
            f.write(str(settings_dict[ky]) + '\n')
        f.close()

    df_summary = pd.DataFrame(columns=headers)

    visited = []

    for session_query in session_queries:

        session_name = session_query['data_name']
        print('data name', session_name)
        if session_name in visited:
            continue

        visited.append(session_name)

        session_data = uow.data.get(session_query['schema_ref'], session_query['data_name'])

        ap_stats = get_animal_performance(session_data, root_path, settings_dict)
        (total_area, total_filled, percent, total_distance, speed, bin_distances, bin_coverage) = ap_stats

        # Build the overall performance row.
        row = {
            "signature": session_name,
            "total_area": total_area,
            "total_filled": total_filled,
            "coverage": percent,
            "total_distance": total_distance,
            "min_speed": np.min(speed),
            "max_speed": np.max(speed),
            "mean_speed": np.mean(speed),
            "median_speed": np.median(speed)
        }
        centre_distance = 0.0
        centre_coverage = 0.0
        boundaries_distance = 0.0
        boundaries_coverage = 0.0
        num_bins = bin_distances.shape[0]  # should be 4
        for i in range(num_bins):
            for j in range(num_bins):
                row[f"bin_distance_r{i+1}_c{j+1}"] = bin_distances[i, j]
                row[f"bin_coverage_r{i+1}_c{j+1}"] = bin_coverage[i, j]
                if i in [1, 2] and j in [1, 2]:
                    centre_distance += bin_distances[i, j]
                    centre_coverage += bin_coverage[i, j]
                else:
                    boundaries_distance += bin_distances[i, j]
                    boundaries_coverage += bin_coverage[i, j]

        centre_coverage /= 4.0 ## average over the 4 centre bins
        boundaries_coverage /= 12.0 ## average over the 12 boundary bins

        row["centre_distance"] = centre_distance
        row["centre_coverage"] = centre_coverage
        row["boundaries_distance"] = boundaries_distance
        row["boundaries_coverage"] = boundaries_coverage

        df_summary.loc[len(df_summary)] = row

    # Save dataframe to CSV
    summary_csv_path = os.path.join(root_path, 'performance_summary.csv')
    df_summary.to_csv(summary_csv_path, index=False)

    print(f'Summary data saved to {summary_csv_path}')


In [ ]:
import time

save_dir = pathlib.Path.cwd().parent / r"data" / r"output"

animal = {'animal_id': '1-13', 'species': 'mouse', 'sex': 'F', 'age': 1, 'weight': 1, 'genotype': 'type', 'animal_notes': 'notes'}
devices = {'axona_led_tracker': True, 'implant': True}
implant = {'implant_id': '001', 'implant_type': 'tetrode', 'implant_geometry': 'square', 'wire_length': 25, 'wire_length_units': 'um', 'implant_units': 'uV'}

csv_header = {}
plotTasks = {}
csv_header_keys = ['signature', 'total_area', 'total_filled', 'coverage', 'min_speed',
                    'max_speed', 'mean_speed', 'median_speed', 'total_distance', "centre_distance",
                    "centre_coverage", "boundaries_distance", "boundaries_coverage"]

num_bins = 4
for i in range(1, num_bins+1):
    for j in range(1, num_bins+1):
        csv_header_keys.append(f"bin_distance_r{i}_c{j}")
for i in range(1, num_bins+1):
    for j in range(1, num_bins+1):
        csv_header_keys.append(f"bin_coverage_r{i}_c{j}")
                    
for key in csv_header_keys:
    csv_header[key] = True

session_settings = {'channel_count': 4, 'animal': animal, 'devices': devices, 'implant': implant}
settings = {'ppm': None, 'session':  session_settings, 'smoothing_factor': 2, 'useMatchedCut': False}

tasks = {}
tasks['disk_arena'] = False # -->
settings['tasks'] = tasks # --> change tasks array to change tasks are run
settings['plotTasks'] = plotTasks # --> change plot tasks array to change asks taht are plotted
settings['header'] = csv_header # --> change csv_header header to change tasks that are saved to csv

""" FOR YOU TO EDIT """
settings['arena_size'] = (50,50)
settings['speed_lowerbound'] = 3
settings['speed_upperbound'] = 100
settings['end_cell'] = None
settings['start_cell'] = None

start_time = time.time()
# root = tk.Tk()
# root.withdraw()
# data_dir = filedialog.askdirectory(parent=root,title='Please select a data directory.')

########################################################################################################################
with unit_of_work as uow:
    query = {
        "schema_ref": "animal_position"
    }
    sorted_by = [("session_start", -1)]
    sessions = uow.data.find(query)
    batch_map_via_queries(uow, sessions, settings, save_dir)
